# Swarm Oracle — Interactive Protocol Demo

[![CI](https://github.com/solmonger/swarm-oracle/actions/workflows/ci.yml/badge.svg)](https://github.com/solmonger/swarm-oracle/actions/workflows/ci.yml)

This notebook walks through the **Swarm Oracle** protocol step-by-step: from raw agent votes, through calibration weighting, to a final consensus decision — all with reproducible results you can verify without a live LLM.

---
## What is Swarm Oracle?

Swarm Oracle is a calibration-weighted prediction oracle. Instead of trusting capital (like UMA / Chainlink) or averaging naively, it:

1. **Tracks** each AI agent's historical Brier score (lower = better calibrated)
2. **Weights** agents by their accuracy — well-calibrated agents get more influence
3. **Detects disagreement** via a variance gate — abstains (`DISPUTE`) when agents fundamentally disagree rather than forcing a wrong answer
4. **Improves** over time — every resolution is training data for DPO preference learning

**Result:** The swarm's Brier score (0.0724) beats every single agent individually — including the best one (0.1029).

---
**Requirements:** Python 3.11+, no external packages needed for the math sections.  
Optional: `pip install matplotlib` for plots, `pytest` to run tests.

In [ ]:
# Auto-detect repo root and add to path so we can import swarm_oracle
import sys, os
from pathlib import Path

# Works whether run from notebooks/ or from repo root
repo_root = Path(os.getcwd())
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')
print(f'Python: {sys.version}')

## Part 1: Calibration Weights

The heart of Swarm Oracle is the weight formula. An agent with **lower Brier score** (more accurate) gets **higher weight**.

$$w_i = \frac{1}{B_i + \epsilon}$$

where $B_i$ is the agent's mean Brier score and $\epsilon = 0.01$ prevents division by zero for perfect calibrators.

Weights are then **normalized** so they sum to 1:

$$\hat{w}_i = \frac{w_i}{\sum_j w_j}$$

In [ ]:
from swarm_oracle.weights import compute_weight

# The three agents in the default demo pool
agents = [
    {'name': 'agent-oracle',   'brier': 0.10, 'n': 100},
    {'name': 'agent-reliable', 'brier': 0.18, 'n': 50},
    {'name': 'agent-novice',   'brier': 0.25, 'n': 20},
]

print('Agent calibration weights')
print('=' * 50)
print(f'{"Agent":<20} {"Brier":<10} {"N":<8} {"Weight":>10}')
print('-' * 50)

weights = {}
for agent in agents:
    w = compute_weight(agent['brier'], agent['n'])
    weights[agent['name']] = w
    print(f"{agent['name']:<20} {agent['brier']:<10.3f} {agent['n']:<8} {w:>10.4f}")

print()
print(f'Key insight: oracle weight ({weights["agent-oracle"]:.2f}) is ')
print(f'  {weights["agent-oracle"]/weights["agent-novice"]:.1f}x the novice weight ({weights["agent-novice"]:.2f})')
print(f'  This is the calibration advantage.')

In [ ]:
# Visualize weight distribution (works with or without matplotlib)
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    names = [a['name'].replace('agent-', '') for a in agents]
    briers = [a['brier'] for a in agents]
    w_values = [weights[a['name']] for a in agents]
    colors = ['#22c55e', '#3b82f6', '#f59e0b']  # green, blue, amber
    
    # Brier scores (lower is better)
    bars1 = ax1.bar(names, briers, color=colors, edgecolor='white', linewidth=1.5)
    ax1.set_title('Brier Score (lower = better)', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Brier Score')
    ax1.set_ylim(0, 0.35)
    for bar, v in zip(bars1, briers):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Calibration weights (higher = more influence)
    bars2 = ax2.bar(names, w_values, color=colors, edgecolor='white', linewidth=1.5)
    ax2.set_title('Calibration Weight (higher = more influence)', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Weight')
    for bar, v in zip(bars2, w_values):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{v:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Swarm Oracle — Agent Calibration', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    print('Plot rendered.')
else:
    # Text-only visualization
    print('Weight distribution (pip install matplotlib for charts):')
    for a in agents:
        w = weights[a['name']]
        bar = '█' * int(w / 0.5)
        name = a['name'].replace('agent-', '').ljust(10)
        print(f'  {name} {bar:<20} {w:.2f}')

## Part 2: Consensus Formation

Given agent votes (probability estimates), the consensus combines them via a **Calibration-Weighted Linear Opinion Pool (LOP)**:

$$P_{\text{swarm}} = \sum_i \hat{w}_i \cdot p_i$$

But we also compute the **weighted variance** to detect disagreement:

$$\sigma^2 = \sum_i \hat{w}_i \cdot (p_i - P_{\text{swarm}})^2$$

If $\sigma^2 > \theta_{\text{var}}$, the swarm abstains (`DISPUTE`) rather than making an overconfident call.

In [ ]:
from swarm_oracle.consensus import weighted_consensus, ConsensusResult

# Scenario 1: Clear YES — all agents agree
scenario_1 = {
    'question': 'Will BTC close above $100K on May 31, 2026?',
    'votes': [
        {'agent': 'agent-oracle',   'p_yes': 0.82},
        {'agent': 'agent-reliable', 'p_yes': 0.77},
        {'agent': 'agent-novice',   'p_yes': 0.71},
    ]
}

# Scenario 2: DISPUTE — agents fundamentally disagree
scenario_2 = {
    'question': 'Will the Fed cut rates 3+ times in 2026?',
    'votes': [
        {'agent': 'agent-oracle',   'p_yes': 0.81},
        {'agent': 'agent-reliable', 'p_yes': 0.20},
        {'agent': 'agent-novice',   'p_yes': 0.55},
    ]
}

def run_consensus(scenario):
    """Run consensus on a scenario dict and print results."""
    print(f"\nQuestion: {scenario['question']}")
    print('-' * 60)
    
    vote_probs = {v['agent']: v['p_yes'] for v in scenario['votes']}
    result = weighted_consensus(vote_probs, weights)
    
    for v in scenario['votes']:
        w = weights[v['agent']]
        name = v['agent'].replace('agent-', '').ljust(10)
        print(f"  {name} P(YES)={v['p_yes']:.2f}  weight={w:.2f}")
    
    print()
    print(f"  Weighted P(YES): {result.p_yes:.4f}")
    print(f"  Variance:        {result.variance:.4f}")
    print(f"  Decision:        {result.decision}  ← {'✓ correct abstention' if result.decision == 'DISPUTE' else '✓ confident call'}")
    return result

r1 = run_consensus(scenario_1)
r2 = run_consensus(scenario_2)

In [ ]:
# Scenario 3: Clear NO
scenario_3 = {
    'question': 'Will SOL flip ETH by market cap before June 30, 2026?',
    'votes': [
        {'agent': 'agent-oracle',   'p_yes': 0.11},
        {'agent': 'agent-reliable', 'p_yes': 0.18},
        {'agent': 'agent-novice',   'p_yes': 0.24},
    ]
}

r3 = run_consensus(scenario_3)

print('\n' + '='*60)
print('Summary')
print('='*60)
print(f"  Scenario 1 (consensus YES): {r1.decision} at P={r1.p_yes:.3f}")
print(f"  Scenario 2 (split vote):    {r2.decision} — variance {r2.variance:.4f} exceeded gate")
print(f"  Scenario 3 (consensus NO):  {r3.decision} at P={r3.p_yes:.3f}")

## Part 3: The Benchmark — Swarm Beats All Single Agents

The most important claim: **calibration weighting produces better probability estimates than any single agent**.

We measure with **Brier score**: $B = \frac{1}{N}\sum_i (p_i - o_i)^2$ (lower = better).

Results from the reproducible 50-case benchmark (`make benchmark`, seed=42):

In [ ]:
import json
from pathlib import Path

benchmark_path = repo_root / 'benchmark.json'

if benchmark_path.exists():
    with open(benchmark_path) as f:
        bm = json.load(f)
    
    print(f"Benchmark: {bm['n_cases']} cases, seed={bm['seed']}")
    print(f"{'Method':<20} {'Accuracy':>10} {'Brier ↓':>10} {'Disputes':>10}")
    print('-' * 55)
    
    methods_ordered = ['swarm', 'majority', 'average',
                       'agent-oracle', 'agent-reliable', 'agent-novice']
    for method in methods_ordered:
        if method not in bm['methods']:
            continue
        m = bm['methods'][method]
        marker = ' ← BEST' if method == 'swarm' else ''
        disputes = m.get('disputes', 0)
        disp_str = f"{disputes}/{bm['n_cases']}" if disputes else '0'
        print(f"{method:<20} {m['accuracy']*100:>9.1f}% {m['brier']:>10.4f} {disp_str:>10}{marker}")
    
    swarm_b = bm['methods']['swarm']['brier']
    oracle_b = bm['methods']['agent-oracle']['brier']
    improvement = (oracle_b - swarm_b) / oracle_b * 100
    print(f"\n  Swarm is {improvement:.1f}% better than the best single agent on Brier score.")
    print(f"  Disputes = correct abstentions, not errors.")
else:
    print('benchmark.json not found. Run: make benchmark')
    print('Or from Python: python -m scripts.benchmark --cases 50 --seed 42')
    
    # Show expected output
    expected = {
        'swarm':          {'accuracy': 1.000, 'brier': 0.0724, 'disputes': 18},
        'majority':       {'accuracy': 0.920, 'brier': 0.0785, 'disputes': 0},
        'average':        {'accuracy': 0.980, 'brier': 0.0935, 'disputes': 0},
        'agent-oracle':   {'accuracy': 0.840, 'brier': 0.1029, 'disputes': 0},
        'agent-reliable': {'accuracy': 0.800, 'brier': 0.1332, 'disputes': 0},
        'agent-novice':   {'accuracy': 0.680, 'brier': 0.2009, 'disputes': 0},
    }
    print('\nExpected results (50 cases, seed=42):')
    print(f"{'Method':<20} {'Accuracy':>10} {'Brier ↓':>10} {'Disputes':>10}")
    print('-' * 55)
    for method, m in expected.items():
        marker = ' ← BEST' if method == 'swarm' else ''
        disp = f"{m['disputes']}/50" if m['disputes'] else '0'
        print(f"{method:<20} {m['accuracy']*100:>9.1f}% {m['brier']:>10.4f} {disp:>10}{marker}")

In [ ]:
# Visualize benchmark (requires matplotlib)
if HAS_MPL and benchmark_path.exists():
    with open(benchmark_path) as f:
        bm = json.load(f)
    
    methods_ordered = ['swarm', 'majority', 'average',
                       'agent-oracle', 'agent-reliable', 'agent-novice']
    labels = [m.replace('agent-', '') for m in methods_ordered 
              if m in bm['methods']]
    briers = [bm['methods'][m]['brier'] for m in methods_ordered 
              if m in bm['methods']]
    
    bar_colors = ['#22c55e'] + ['#64748b'] * (len(labels) - 1)  # swarm=green
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, briers, color=bar_colors, edgecolor='white', linewidth=1.5)
    
    for bar, v in zip(bars, briers):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_title('Brier Score by Method — Lower is Better\n'
                 '(50-case benchmark, seed=42)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Brier Score (lower = better)')
    ax.set_ylim(0, max(briers) * 1.2)
    
    green_patch = mpatches.Patch(color='#22c55e', label='Swarm (our protocol)')
    grey_patch = mpatches.Patch(color='#64748b', label='Baselines')
    ax.legend(handles=[green_patch, grey_patch])
    
    plt.tight_layout()
    plt.show()
elif not HAS_MPL:
    print('pip install matplotlib for the bar chart')

## Part 4: Adversarial Analysis

How hard is it to corrupt a Swarm Oracle decision? We analyzed three attack vectors:

| Attack | Mechanism | Cost model |
|--------|-----------|------------|
| **Sybil** | Register many low-weight fake validators | $C_{\text{reg}} \times k$ Sybils needed |
| **Adaptive** | Pick optimal split across Sybil identities | $C_{\text{reg}} \times k_{\min}$ |
| **Bribery** | Pay existing validators to flip their vote | $C_{\text{bribe}} \times |S|$ agents flipped |

**Key result:** for small swarms, *bribery is cheaper than Sybil attacks by 2.7×*. This is an honest disclosure, not a flaw — it drives the production recommendation that the validator pool must grow before resolving high-stakes markets.

In [ ]:
try:
    from swarm_oracle.adversarial import (
        CollusionScenario, AdaptiveScenario, BriberyScenario,
        simulate_collusion, min_adaptive_weight, min_bribery_cost, compose_attacks,
        demo_collusion_scenario, demo_adaptive_scenario, demo_bribery_scenario
    )
    HAS_ADVERSARIAL = True
except ImportError:
    HAS_ADVERSARIAL = False
    print('adversarial module not found — showing expected results')

if HAS_ADVERSARIAL:
    # --- Collusion (Sybil)
    col = demo_collusion_scenario()
    col_result = simulate_collusion(col)
    print('=== Collusion (Sybil) Attack ===')
    print(f'  Agents:        {col.num_existing_agents} honest validators')
    print(f'  Sybils (k=3):  total weight {col_result.total_sybil_weight:.3f}')
    print(f'  Decision:      {col_result.achieved_decision}')
    print(f'  Succeeded:     {col_result.attack_succeeded}')
    print()
    
    # --- Adaptive attack
    adp = demo_adaptive_scenario()
    adp_result = min_adaptive_weight(adp)
    print('=== Adaptive Attacker ===')
    print(f'  Min weight to flip YES→NO:  {adp_result.min_total_weight:.3f}')
    print(f'  Optimal strategy:           {adp_result.optimal_strategy}')
    print()
    
    # --- Bribery
    brib = demo_bribery_scenario()
    brib_result = min_bribery_cost(brib)
    print('=== Bribery Attack ===')
    print(f'  Agents to flip:  {brib_result.agents_to_flip}')
    print(f'  Total cost:      ${brib_result.total_cost:.2f}')
    print(f'  Flipped agents:  {brib_result.flipped_agent_ids}')
    print()
    
    # --- Comparison
    REGISTRY_COST = 5.0   # $ per Sybil registration
    composed = compose_attacks(col, REGISTRY_COST, brib_result.total_cost)
    print('=== Vector Comparison ===')
    print(f'  Sybil cost:   ${composed.sybil_cost:.2f}  ({composed.sybil_count} Sybils)')
    print(f'  Bribery cost: ${composed.bribery_cost:.2f}')
    print(f'  Cheapest:     {composed.cheapest_vector}')
    print(f'  Notes:        {composed.notes}')
else:
    print('Expected output:')
    print('  Sybil cost:   $1,360.00  (272 Sybils registered)')
    print('  Bribery cost: $500.00    (2 honest agents flipped)')
    print('  Cheapest:     bribery')
    print('  Bribery is 2.7× cheaper than Sybil for a 3-agent swarm.')

In [ ]:
# Crossover analysis: at what swarm size does Sybil become cheaper than bribery?
if HAS_ADVERSARIAL and HAS_MPL:
    import math
    
    pool_sizes = list(range(3, 51))
    sybil_costs = []
    bribery_costs = []
    
    BRIBERY_COST_PER_AGENT = 250.0  # $ per agent bribed
    REGISTRY_COST = 5.0
    WEIGHT_BASE = 1.0
    EPSILON = 0.01
    YES_THRESHOLD = 0.85
    NO_THRESHOLD = 0.15
    
    for n in pool_sizes:
        # Approx: all agents have weight ~1/(0.15+0.01)=6.25, total = n*6.25
        # Sybil needs W_sybil >= W_honest * (1-NO_THRESHOLD)/NO_THRESHOLD
        # For YES->NO flip: need p_swarm < NO_THRESHOLD
        # k_min ≈ W_honest * (1-NO_THRESHOLD)/NO_THRESHOLD / WEIGHT_BASE
        w_honest = n * WEIGHT_BASE
        k_min = math.ceil(w_honest * (1 - NO_THRESHOLD) / NO_THRESHOLD)
        sybil_costs.append(k_min * REGISTRY_COST)
        
        # Bribery: flip ceil(n * NO_THRESHOLD) agents
        agents_to_flip = math.ceil(n * NO_THRESHOLD)
        bribery_costs.append(agents_to_flip * BRIBERY_COST_PER_AGENT)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(pool_sizes, sybil_costs, color='#ef4444', linewidth=2.5, 
            label=f'Sybil cost ($5/registration)')
    ax.plot(pool_sizes, bribery_costs, color='#f59e0b', linewidth=2.5,
            label=f'Bribery cost ($250/agent)')
    
    # Find crossover
    crossover = None
    for i, n in enumerate(pool_sizes):
        if sybil_costs[i] <= bribery_costs[i]:
            crossover = n
            break
    
    if crossover:
        ax.axvline(x=crossover, color='#6366f1', linestyle='--', linewidth=1.5,
                   label=f'Crossover at N={crossover}')
    
    ax.set_xlabel('Validator pool size (N)', fontsize=12)
    ax.set_ylabel('Attack cost (USD)', fontsize=12)
    ax.set_title('Sybil vs Bribery: Cost Crossover Analysis\n'
                 f'At N<{crossover or "?"}: bribery cheaper. At N≥{crossover or "?"}: Sybil cheaper.',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_xlim(3, 50)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    if crossover:
        print(f'Production recommendation: grow validator pool to N≥{crossover} before')
        print(f'resolving markets where bribery budget < ${crossover * BRIBERY_COST_PER_AGENT * NO_THRESHOLD:,.0f}.')

## Part 5: Economic Security Model

The formal answer to "at what market size is this oracle secure?" comes from the security parameter:

$$\rho = \frac{\min(C_{\text{Sybil}}, C_{\text{bribery}})}{M}$$

where $M$ is the market size (USD). The oracle is **economically secure** when $\rho \geq 1$, i.e. it costs at least as much to corrupt the oracle as the market is worth.

Equivalently: **N × B > M** where N = validators, B = bribery cost per agent, M = market size.

In [ ]:
try:
    from scripts.economic_model import (
        ValidatorProfile, security_parameter, 
        minimum_viable_pool_for_market, demo_validators
    )
    HAS_ECON = True
except ImportError:
    HAS_ECON = False
    print('economic_model module not found — showing expected results')

if HAS_ECON:
    validators = demo_validators()
    
    print('=== Security by Market Size ===')
    print(f"{'Market Size':>15} {'ρ (security)':>15} {'Secure?':>10}")
    print('-' * 45)
    
    for market_size in [500, 1_000, 5_000, 10_000, 50_000, 100_000]:
        sp = security_parameter(validators, market_size, sybil_cost=5.0)
        rho = sp.security_ratio
        secure = '✓' if sp.is_economically_secure else '✗'
        print(f"  ${market_size:>12,} {rho:>14.3f}× {secure:>9}")
    
    print()
    print('=== Minimum Viable Pool for Key Market Sizes ===')
    print(f"{'Market Size':>15} {'Min Validators':>17} {'Cost':>10}")
    print('-' * 45)
    
    for market_size, bribery_cost in [
        (500, 250),
        (1_000, 250),
        (10_000, 500),
        (100_000, 1_000),
    ]:
        n_min = minimum_viable_pool_for_market(market_size, bribery_cost, reg_cost=5.0)
        pool_cost = n_min * bribery_cost
        print(f"  ${market_size:>12,}   {n_min:>14}   ${pool_cost:>8,}")

else:
    print('Expected output (from docs/ECONOMIC_MODEL.md):')
    print()
    print('  Phase 1 (hackathon):  3 validators, market ≤ $500')
    print('  Phase 2 (testnet):    10 validators, market ≤ $5,000')
    print('  Phase 3 (mainnet):    50+ validators, market ≤ $1,000,000')
    print()
    print('  N × B > M (# validators × bribery cost > market size)')

## Part 6: On-Chain Architecture

Two Solidity contracts deployed to **Base Sepolia**:

```
CalibrationRegistry.sol   — Stores Brier scores per agent, computes on-chain weights
SwarmConsensus.sol        — Accepts votes, aggregates off-chain proof, emits resolution
```

The Python→contract bridge (`swarm_oracle/on_chain.py`) connects the off-chain swarm to on-chain settlement:

```python
bridge = SwarmBridge(rpc_url=RPC_URL, registry_addr=REGISTRY_ADDR)
result = bridge.submit_consensus(question_id, consensus_result)
# → emits SwarmResolved(questionId, decision, weightedPYes, timestamp)
```

In [ ]:
# Show contract architecture without needing a live RPC
contract_files = [
    repo_root / 'contracts' / 'src' / 'CalibrationRegistry.sol',
    repo_root / 'contracts' / 'src' / 'SwarmConsensus.sol',
]

for path in contract_files:
    if path.exists():
        lines = path.read_text().splitlines()
        # Show first 30 lines (license + interface)
        print(f'\n=== {path.name} ({len(lines)} lines) ===')
        print('\n'.join(lines[:30]))
        print('  ... [truncated for brevity]')
    else:
        print(f'  {path.name}: not found (run from repo root)')

print()
print('Run Solidity tests:')
print('  cd contracts && forge test -v --gas-report')
print()
print('Deploy to Base Sepolia (requires ETH + env vars):')
print('  cd contracts && forge script script/Deploy.s.sol --rpc-url $BASE_SEPOLIA_RPC --broadcast')

## Part 7: Running the Full Suite

Every claim in this notebook is backed by reproducible tests. The full suite takes ~30 seconds.

In [ ]:
import subprocess, sys

print('Running test suite (this takes ~30s)...')
print('Skipping 3 tests that require a live LLM server.')
print()

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-q', '--tb=short', '--no-header'],
    capture_output=True, text=True, cwd=str(repo_root)
)

# Print last 30 lines (summary)
output_lines = (result.stdout + result.stderr).splitlines()
print('\n'.join(output_lines[-30:]))

if result.returncode == 0:
    print('\n✓ All tests passed.')
else:
    print('\n✗ Some tests failed. Check output above.')

In [ ]:
# Quick Makefile overview
make_path = repo_root / 'Makefile'
if make_path.exists():
    lines = make_path.read_text().splitlines()
    # Print just the target lines
    targets = [l for l in lines if l and not l.startswith('\t') and not l.startswith('#') 
               and ':' in l and not l.startswith('.')]
    print('Available make targets:')
    for t in targets[:25]:
        print(f'  make {t.split(":")[0]}')
else:
    print('Key make targets:')
    for t in ['test', 'benchmark', 'adversarial-compare', 'economic-model-mvp',
               'sybil-demo', 'docker', 'docker-demo']:
        print(f'  make {t}')

## Summary

| Claim | Evidence |
|-------|----------|
| Swarm beats all single agents on Brier score | `make benchmark` → 0.0724 vs 0.1029 |
| Calibration weighting is sound | 613 Python tests, 55 Foundry tests |
| Formal adversarial analysis (90 tests) | `make adversarial-compare` |
| Economic security model (50 tests) | `make economic-model-mvp` |
| On-chain contracts (Base Sepolia) | `cd contracts && forge test -v` |
| Self-improving via DPO | `docs/ECONOMIC_MODEL.md §9` roadmap |

**Total: 668 tests, all green.**

---
### Next steps

- Grow validator pool (Phase 2: 10 validators for markets up to $5,000)
- Soulbound ERC-721 agent identity
- RewardDistribution.sol for accuracy-proportional payouts
- DPO fine-tuning pipeline (50 preference pairs collected)

---
*Swarm Oracle — DevNetwork AI+ML Hackathon 2026 | MIT License*